# Indra OCR quickstart

Extract text from an image file, and from an image already in memory. The first run downloads the models from Hugging Face (about 76 MB).

In [ ]:
%pip install -q indra-ocr

In [ ]:
import indra_ocr

conv = indra_ocr.load(threads=4)

In [ ]:
# Two sample pages, if you have no scan.png or equations.png of your own
import os, urllib.request
BASE = "https://raw.githubusercontent.com/datalab-to/indra/main/examples/samples/"
for name in ("scan.png", "equations.png"):
    if not os.path.exists(name):
        urllib.request.urlretrieve(BASE + name, name)

## An image file

`convert` reads images (PNG, JPEG, TIFF, BMP, WebP) and PDFs. It yields `(page index, Page)`; a multi-page TIFF gives one page per frame.

In [ ]:
for i, page in conv.convert("scan.png"):
    print(page.markdown)   # headings, lists, tables as HTML
    print(page.text)       # plain text, one line per line

## Lines, boxes and layout

Every line has a box in image pixels, its text and a confidence, in reading order. `page.blocks` holds the layout blocks (label, box, text).

In [ ]:
for ln in page.lines[:10]:
    print(ln.box, round(ln.conf, 2), ln.text)

for b in page.blocks[:5]:
    print(b["label"], b["bbox"])

## An image in memory

Pass a grayscale `uint8` numpy array to `conv.ocr`, for example a camera frame, a crop or a decoded upload.

In [ ]:
from PIL import Image
import numpy as np  # Pillow ships with indra-ocr

color = np.asarray(Image.open("scan.png"))     # or any uint8 RGB / BGR array
gray = np.asarray(Image.fromarray(color).convert("L"))
page = conv.ocr(gray)
print(page.text)
print("turned", page.rotation, "degrees, straightened", page.skew, "degrees")

## Math

With `math=True`, lines with math come out as LaTeX (slower; the math models download on first use).

In [ ]:
conv_math = indra_ocr.load(threads=4, math=True)
for i, page in conv_math.convert("equations.png"):
    print(page.markdown)